# CivicGuard AI MobileNetV2 Training Workflow

This notebook trains a transfer-learning classifier for the CivicGuard AI hazard classes:

- blocked_drain
- sewage_overflow
- road_damage
- fallen_tree
- water_logging

The notebook is designed to run in Google Colab or a local notebook environment with TensorFlow installed.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

DATA_ROOT = Path(os.environ.get('DATA_ROOT', 'data/processed'))
OUTPUT_DIR = Path(os.environ.get('OUTPUT_DIR', 'artifacts'))
MODEL_PATH = OUTPUT_DIR / 'mobilenetv2_civicguard.keras'
CLASS_NAMES = ['blocked_drain', 'sewage_overflow', 'road_damage', 'fallen_tree', 'water_logging']
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tf.random.set_seed(SEED)
np.random.seed(SEED)

## Dataset layout

Expected structure:

```text
data/processed/
  blocked_drain/
  sewage_overflow/
  road_damage/
  fallen_tree/
  water_logging/
```

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    DATA_ROOT,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)
val_ds = keras.utils.image_dataset_from_directory(
    DATA_ROOT,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)
class_names = train_ds.class_names
class_names

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False
inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)
model = keras.Model(inputs, outputs)
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

In [ ]:
loss, accuracy = model.evaluate(val_ds)
print({'validation_loss': float(loss), 'validation_accuracy': float(accuracy)})

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.title('Accuracy')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
model.save(MODEL_PATH)
print(f'Saved model to {MODEL_PATH}')

In [ ]:
labels_path = OUTPUT_DIR / 'class_names.json'
labels_path.write_text(json.dumps(class_names, indent=2), encoding='utf-8')
print(f'Saved labels to {labels_path}')